In [1]:
import yaml
import json
import pandas as pd
import numpy as np
import xlsxwriter
from math import floor

__Helper to replicate Excel's conventional rounding methodology__

In [2]:
def conventional_round(n, places=0):
    multiplier = 10 ** places
    return floor(n*multiplier + 0.5) / multiplier

__Step 1: Read data__

In [3]:
#mapping
mapping = pd.read_excel("J:\\EUA\\CRA\\Collateral Review\\Volatility Factor\\HIST\\2023\\python_inputs\\lending_value_python_inputs_2023.xlsx", sheet_name = "mapping", 
    dtype = {"COLLATERAL_PRIORITY": float, "NLL_CODE": str})

#credit admin factors
ca = pd.read_excel("J:\\EUA\\CRA\\Collateral Review\\Volatility Factor\\HIST\\2023\\python_inputs\\lending_value_python_inputs_2023.xlsx", sheet_name = "ca_factors", 
    dtype = {"CREDIT_A": float, "CREDIT_B": float, "MODEL_PRICE": float, "MODEL_UPB": float,
    "LIQUIDATION_PREP": float, "LIQUIDATION_SALE": float})

#volatility factors
vol = pd.read_excel("J:\\EUA\\CRA\\Collateral Review\\Volatility Factor\\HIST\\2023\\python_inputs\\lending_value_python_inputs_2023.xlsx", sheet_name = "vol_factors", 
    dtype = {"10D": float, "84D": float, "252D": float})

#present lending values
lvp = pd.read_excel("J:\\EUA\\CRA\\Collateral Review\\Volatility Factor\\HIST\\2023\\python_inputs\\lending_value_python_inputs_2023.xlsx", sheet_name = "present_lv", 
    dtype = {"LV_A": float, "LV_B": float})

df = pd.merge(mapping, ca, on = "CREDIT_ADMIN_FACTOR_ID", how = "left") #merge mapping with credit admin factors
df = pd.merge(df, lvp, on = ["COLLATERAL_GROUP_CODE", "COLLATERAL_ACE_CODE", "COLLATERAL_TABLE", "FHLB_RATING"], how = "left") #merge dataframe with present lending values

__Step 2: Calculate volatility factors__

In [4]:
vol_dict_a_sec = vol.set_index("VOLATILITY_FACTOR_ID").to_dict()["10D"]
vol_dict_a_loan = vol.set_index("VOLATILITY_FACTOR_ID").to_dict()["84D"]
vol_dict_b = vol.set_index("VOLATILITY_FACTOR_ID").to_dict()["252D"]

#volatility factors
df["VOLATILITY_A"] = np.where(df["COLLATERAL_TABLE"] == "Securities", df["VOLATILITY_FACTOR_ID"].map(vol_dict_a_sec), df["VOLATILITY_FACTOR_ID"].map(vol_dict_a_loan)) #class a
df["VOLATILITY_B"] = df["VOLATILITY_FACTOR_ID"].map(vol_dict_b) #class b

__Step 3: Calculate raw lending values__

In [5]:
#calculated lending values
factorsa = ["CREDIT_A", "CREDIT_B", "MODEL_PRICE", "MODEL_UPB", "LIQUIDATION_PREP", "LIQUIDATION_SALE", "VOLATILITY_A"] #class a
df["PROPOSED_LV_CL_A_RAW"] = (100 - df[factorsa].sum(axis=1)).apply(lambda x: conventional_round(x))

factorsb = ["CREDIT_A", "CREDIT_B", "MODEL_PRICE", "MODEL_UPB", "LIQUIDATION_PREP", "LIQUIDATION_SALE", "VOLATILITY_B"] #class b
df["PROPOSED_LV_CL_B_RAW"] = (100 - df[factorsb].sum(axis=1)).apply(lambda x: conventional_round(x))

#LV change: calculated vs present
#df["CL_A_CHANGE"] = abs(df["PROPOSED_LV_CL_A_RAW"] - df["LV_A"]) #class a; uncomment to reinstitute LV change rules
#df["CL_B_CHANGE"] = abs(df["PROPOSED_LV_CL_B_RAW"] - df["LV_B"]) #class b; uncomment to reinstitute LV change rules

__Step 4: Calculate Adjusted Lending Values__

In [6]:
#LV change rules
#class a
#df.loc[(df["PROPOSED_LV_CL_A_RAW"] < 85) & (df["CL_A_CHANGE"] <= 2), "PROPOSED_LV_CL_A"] = df["LV_A"] #uncomment to reinstitute LV change rules
#df.loc[(df["PROPOSED_LV_CL_A_RAW"] >= 85) | (df["CL_A_CHANGE"] > 2), "PROPOSED_LV_CL_A"] = np.minimum(df["PROPOSED_LV_CL_A_RAW"], 99) #cap LV at 99; uncomment to reinstitute LV change rules
df["PROPOSED_LV_CL_A"] = np.minimum(df["PROPOSED_LV_CL_A_RAW"], 99) #cap LV at 99; comment if LV change rules reinstituted
df.loc[df["COLLATERAL_ACE_CODE"].isin(["CDSXXX", "ONDXXX"]), "PROPOSED_LV_CL_A"] = df["PROPOSED_LV_CL_A_RAW"] #allow LV at 100

#class b
#df.loc[(df["PROPOSED_LV_CL_B_RAW"] < 85) & (df["CL_B_CHANGE"] <= 2), "PROPOSED_LV_CL_B"] = df["LV_B"] #uncomment to reinstitute LV change rules
#df.loc[(df["PROPOSED_LV_CL_B_RAW"] >= 85) | (df["CL_B_CHANGE"] > 2), "PROPOSED_LV_CL_B"] = np.minimum(df["PROPOSED_LV_CL_B_RAW"], 99) #cap LV at 99; uncomment to reinstitute LV change rules
df["PROPOSED_LV_CL_B"] = np.minimum(df["PROPOSED_LV_CL_B_RAW"], 99) #cap LV at 99; comment if LV change rules reinstituted
df.loc[df["COLLATERAL_ACE_CODE"].isin(["CDSXXX", "ONDXXX"]), "PROPOSED_LV_CL_B"] = df["PROPOSED_LV_CL_B_RAW"] #allow LV at 100

#non-lead lender
df["NLL_KEY"] = np.where(df["NLL_CODE"] == "0", None, df["NLL_CODE"] + "_" + df["COLLATERAL_TABLE"]) #concatenate columns for NLL key

df2 = df.loc[(df["NLL_CODE"] == "0") & (df["COLLATERAL_TABLE"] != "Securities"), ["COLLATERAL_ACE_CODE", "COLLATERAL_TABLE", "PROPOSED_LV_CL_A", "PROPOSED_LV_CL_B"]].copy()
df2["NLL_KEY"] = df2["COLLATERAL_ACE_CODE"] + "_" + df2["COLLATERAL_TABLE"] #concatenate columns for NLL key

dfa = df2[["NLL_KEY", "PROPOSED_LV_CL_A"]]
dfb = df2[["NLL_KEY", "PROPOSED_LV_CL_B"]]

nll_dict_a = dfa.set_index("NLL_KEY").to_dict()["PROPOSED_LV_CL_A"]
nll_dict_b = dfb.set_index("NLL_KEY").to_dict()["PROPOSED_LV_CL_B"]

#NLL LV 10 less than standard
df["NLL_A"] = df["NLL_KEY"].map(nll_dict_a) -10 #class a
df["NLL_B"] = df["NLL_KEY"].map(nll_dict_b) -10 #class b

#floor NLL LV at 0
df["NLL_A"] = np.maximum(df["NLL_A"], 0) #class a
df["NLL_B"] = np.maximum(df["NLL_B"], 0) #class b

df.loc[~df["NLL_KEY"].isnull(), "PROPOSED_LV_CL_A"] = df["NLL_A"]
df.loc[~df["NLL_KEY"].isnull(), "PROPOSED_LV_CL_B"] = df["NLL_B"]

df["PRESENT_LV_CL_A"] = df["LV_A"]
df["PRESENT_LV_CL_B"] = df["LV_B"]

#LV change: calculated vs present
df["CHANGE_CL_A"] = df["PROPOSED_LV_CL_A"] - df["LV_A"] #class a
df["CHANGE_CL_B"] = df["PROPOSED_LV_CL_B"] - df["LV_B"] #class b

#factors to 1 decimal place for display
df["CREDIT_A"] = round(df["CREDIT_A"],1)
df["CREDIT_B"] = round(df["CREDIT_B"],1)
df["MODEL_PRICE"] = round(df["MODEL_PRICE"],1)
df["MODEL_UPB"] = round(df["MODEL_UPB"],1)
df["LIQUIDATION_PREP"] = round(df["LIQUIDATION_PREP"],1)
df["LIQUIDATION_SALE"] = round(df["LIQUIDATION_SALE"],1)
df["VOLATILITY_A"] = round(df["VOLATILITY_A"],1)
df["VOLATILITY_B"] = round(df["VOLATILITY_B"],1)

#management override of LV
#df.loc[(df["COLLATERAL_ACE_CODE"] == "MFXXXX") & (df["COLLATERAL_TABLE"] == "Limited"), "PROPOSED_LV_CL_A"] = 70 #uncomment to apply management override of LV
#df.loc[(df["COLLATERAL_ACE_CODE"] == "MFXXXX") & (df["COLLATERAL_TABLE"] == "QCD"), "PROPOSED_LV_CL_A"] = 70 #uncomment to apply management override of LV

#df.loc[(df["COLLATERAL_ACE_CODE"] == "MFXXXX") & (df["COLLATERAL_TABLE"] == "Limited"), "PROPOSED_LV_CL_B"] = 67 #uncomment to apply management override of LV
#df.loc[(df["COLLATERAL_ACE_CODE"] == "MFXXXX") & (df["COLLATERAL_TABLE"] == "QCD"), "PROPOSED_LV_CL_B"] = 67 #uncomment to apply management override of LV

__Step 5: Duplicate df__

In [ ]:
#loan
df_loancs = df.loc[(df["COLLATERAL_TABLE"] != "Securities"), ["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY", "COLLATERAL_ACE_CODE", "ACE_CODE_DESC", "COLLATERAL_TABLE", "PROPOSED_LV_CL_A", "PROPOSED_LV_CL_B", "CAP", "CDFI"]].copy()

#securities
df_seccs = df.loc[(df["COLLATERAL_TABLE"] == "Securities"), ["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY", "COLLATERAL_ACE_CODE", "ACE_CODE_DESC", "FHLB_RATING", "PROPOSED_LV_CL_A", "PROPOSED_LV_CL_B", "MDA_REVIEW_REQ", "CASE_BY_CASE_FEE"]].copy()
df_seccs["BLANKET_CL_A"] = df_seccs["PROPOSED_LV_CL_A"]
df_seccs["SPECIFIC_CL_A"] = df_seccs["PROPOSED_LV_CL_A"]
df_seccs["SPECIFIC_CL_B"] = df_seccs["PROPOSED_LV_CL_B"]

#MPP
df_MPP = df.copy()

__Step 6: Create exhibit A_detail__

In [ ]:
df = df.sort_values(["COLLATERAL_PRIORITY", "COLLATERAL_ACE_CODE", "FHLB_RATING"], ascending=[True, True, False])

#duplicate df for table tabs
df_exp = df.loc[(df["COLLATERAL_TABLE"] == "Expanded")].copy()
df_ltd = df.loc[(df["COLLATERAL_TABLE"] == "Limited")].copy()
df_qcd = df.loc[(df["COLLATERAL_TABLE"] == "QCD")].copy()
df_war = df.loc[(df["COLLATERAL_TABLE"] == "Warehouse")].copy()
df_sec = df.loc[(df["COLLATERAL_TABLE"] == "Securities")].copy()

#insert empty row at each change of ACE_Group
df = df.reset_index(drop=True) #exhibit a
mask = df["COLLATERAL_GROUP_CODE"].ne(df["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df.columns)
df = pd.concat([df, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df = df.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

df_exp = df_exp.reset_index(drop=True) #expanded
mask = df_exp["COLLATERAL_GROUP_CODE"].ne(df_exp["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_exp.columns)
df_exp = pd.concat([df_exp, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_exp = df_exp.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

df_ltd = df_ltd.reset_index(drop=True) #limited
mask = df_ltd["COLLATERAL_GROUP_CODE"].ne(df_ltd["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_ltd.columns)
df_ltd = pd.concat([df_ltd, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_ltd = df_ltd.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

df_qcd = df_qcd.reset_index(drop=True) #qcd
mask = df_qcd["COLLATERAL_GROUP_CODE"].ne(df_qcd["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_qcd.columns)
df_qcd = pd.concat([df_qcd, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_qcd = df_qcd.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

df_war = df_war.reset_index(drop=True) #warehouse
mask = df_war["COLLATERAL_GROUP_CODE"].ne(df_war["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_war.columns)
df_war = pd.concat([df_war, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_war = df_war.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

df_sec = df_sec.reset_index(drop=True) #securities
mask = df_sec["COLLATERAL_GROUP_CODE"].ne(df_sec["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_sec.columns)
df_sec = pd.concat([df_sec, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_sec = df_sec.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

#write exhibit a to .xlsx
cols_det = ["COLLATERAL_ACE_CODE", "ACE_CODE_DESC", "COLLATERAL_TABLE", "FHLB_RATING", "CREDIT_ADMIN_FACTOR_ID", "VOLATILITY_FACTOR_ID", "CREDIT_A", "CREDIT_B", "MODEL_PRICE", "MODEL_UPB", "LIQUIDATION_PREP", "LIQUIDATION_SALE", "VOLATILITY_A", "VOLATILITY_B", "PROPOSED_LV_CL_A", "PROPOSED_LV_CL_B", "PRESENT_LV_CL_A", "PRESENT_LV_CL_B", "CHANGE_CL_A", "CHANGE_CL_B"]
cols_fs = ["COLLATERAL_ACE_CODE", "ACE_CODE_DESC", "COLLATERAL_TABLE", "FHLB_RATING", "CREDIT_ADMIN_FACTOR_ID", "VOLATILITY_FACTOR_ID", "PROPOSED_LV_CL_A", "PROPOSED_LV_CL_B"]

header1 = '&LExhibit_A'
header2 = '&LExhibit_A_Expanded'
header3 = '&LExhibit_A_Limited'
header4 = '&LExhibit_A_QCD'
header5 = '&LExhibit_A_Warehouse'
header6 = '&LExhibit_A_Securities'

writer_det = pd.ExcelWriter("J:\\EUA\\CRA\\Collateral Review\\Volatility Factor\\HIST\\2023\\output\\Exhibit_A_Detail.xlsx", engine='xlsxwriter')
df[cols_det].to_excel(writer_det, sheet_name='Exhibit_A')
df_exp[cols_det].to_excel(writer_det, sheet_name='Expanded')
df_ltd[cols_det].to_excel(writer_det, sheet_name='Limited')
df_qcd[cols_det].to_excel(writer_det, sheet_name='QCD')
df_war[cols_det].to_excel(writer_det, sheet_name='Warehouse')
df_sec[cols_det].to_excel(writer_det, sheet_name='Securities')

workbook = writer_det.book
worksheet1 = writer_det.sheets['Exhibit_A']
worksheet2 = writer_det.sheets['Expanded']
worksheet3 = writer_det.sheets['Limited']
worksheet4 = writer_det.sheets['QCD']
worksheet5 = writer_det.sheets['Warehouse']
worksheet6 = writer_det.sheets['Securities']

cell_format = workbook.add_format({'border': 1, 'border_color': 'black', 'font_name': 'calibri', 'font_size': 10, 'align': 'center', 'valign': 'vcenter'})

worksheet1.set_column('A:W', None, cell_format)
worksheet2.set_column('A:W', None, cell_format)
worksheet3.set_column('A:W', None, cell_format)
worksheet4.set_column('A:W', None, cell_format)
worksheet5.set_column('A:W', None, cell_format)
worksheet6.set_column('A:W', None, cell_format)

worksheet1.set_header(header1)
worksheet2.set_header(header2)
worksheet3.set_header(header3)
worksheet4.set_header(header4)
worksheet5.set_header(header5)
worksheet6.set_header(header6)

header_format = workbook.add_format({'bold': True})

writer_det.save()

writer_fs = pd.ExcelWriter("J:\\EUA\\CRA\\Collateral Review\\Volatility Factor\\HIST\\2023\\output\\Exhibit_A_FinSvcs.xlsx", engine='xlsxwriter')
df[cols_fs].to_excel(writer_fs, sheet_name='Exhibit_A')
df_exp[cols_fs].to_excel(writer_fs, sheet_name='Expanded')
df_ltd[cols_fs].to_excel(writer_fs, sheet_name='Limited')
df_qcd[cols_fs].to_excel(writer_fs, sheet_name='QCD')
df_war[cols_fs].to_excel(writer_fs, sheet_name='Warehouse')
df_sec[cols_fs].to_excel(writer_fs, sheet_name='Securities')

workbook = writer_fs.book
worksheet1 = writer_fs.sheets['Exhibit_A']
worksheet2 = writer_fs.sheets['Expanded']
worksheet3 = writer_fs.sheets['Limited']
worksheet4 = writer_fs.sheets['QCD']
worksheet5 = writer_fs.sheets['Warehouse']
worksheet6 = writer_fs.sheets['Securities']

cell_format = workbook.add_format({'border': 1, 'border_color': 'black', 'font_name': 'calibri', 'font_size': 10, 'align': 'center', 'valign': 'vcenter'})

worksheet1.set_column('A:K', None, cell_format)
worksheet2.set_column('A:K', None, cell_format)
worksheet3.set_column('A:K', None, cell_format)
worksheet4.set_column('A:K', None, cell_format)
worksheet5.set_column('A:K', None, cell_format)
worksheet6.set_column('A:K', None, cell_format)

worksheet1.set_header(header1)
worksheet2.set_header(header2)
worksheet3.set_header(header3)
worksheet4.set_header(header4)
worksheet5.set_header(header5)
worksheet6.set_header(header6)

header_format = workbook.add_format({'bold': True})

writer_fs.save()

__Step 7: Create loan cheat sheet__

In [ ]:
df_loancs = df_loancs.sort_values(["COLLATERAL_PRIORITY", "COLLATERAL_ACE_CODE"], ascending=[True, True])

#duplicate loan df for table tabs
df_expcs = df_loancs.loc[(df_loancs["COLLATERAL_TABLE"] == "Expanded")].copy()
df_ltdcs = df_loancs.loc[(df_loancs["COLLATERAL_TABLE"] == "Limited")].copy()
df_qcdcs = df_loancs.loc[(df_loancs["COLLATERAL_TABLE"] == "QCD")].copy()
df_warcs = df_loancs.loc[(df_loancs["COLLATERAL_TABLE"] == "Warehouse")].copy()

#insert empty row at each change of ACE_Group
df_loancs = df_loancs.reset_index(drop=True) #loan
mask = df_loancs["COLLATERAL_GROUP_CODE"].ne(df_loancs["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_loancs.columns)
df_loancs = pd.concat([df_loancs, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_loancs = df_loancs.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

df_expcs = df_expcs.reset_index(drop=True) #expanded
mask = df_expcs["COLLATERAL_GROUP_CODE"].ne(df_expcs["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_expcs.columns)
df_expcs = pd.concat([df_expcs, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_expcs = df_expcs.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

df_ltdcs = df_ltdcs.reset_index(drop=True) #limited
mask = df_ltdcs["COLLATERAL_GROUP_CODE"].ne(df_ltdcs["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_ltdcs.columns)
df_ltdcs = pd.concat([df_ltdcs, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_ltdcs = df_ltdcs.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

df_qcdcs = df_qcdcs.reset_index(drop=True) #qcd
mask = df_qcdcs["COLLATERAL_GROUP_CODE"].ne(df_qcdcs["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_qcdcs.columns)
df_qcdcs = pd.concat([df_qcdcs, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_qcdcs = df_qcdcs.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

df_warcs = df_warcs.reset_index(drop=True) #warehouse
mask = df_warcs["COLLATERAL_GROUP_CODE"].ne(df_warcs["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_warcs.columns)
df_warcs = pd.concat([df_warcs, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_warcs = df_warcs.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

#write loan cheat sheet to .xlsx
header1 = '&LLoan_Cheat_Sheet'
header2 = '&LLoan_Cheat_Sheet_Expanded'
header3 = '&LLoan_Cheat_Sheet_Limited'
header4 = '&LLoan_Cheat_Sheet_QCD'
header5 = '&LLoan_Cheat_Sheet_Warehouse'

writer = pd.ExcelWriter("J:\\EUA\\CRA\\Collateral Review\\Volatility Factor\\HIST\\2023\\output\\Loan_Cheat_Sheet_FinSvcs.xlsx", engine='xlsxwriter')
df_loancs.to_excel(writer, sheet_name='Loan_Cheat_Sheet')
df_expcs.to_excel(writer, sheet_name='Expanded')
df_ltdcs.to_excel(writer, sheet_name='Limited')
df_qcdcs.to_excel(writer, sheet_name='QCD')
df_warcs.to_excel(writer, sheet_name='Warehouse')

workbook = writer.book
worksheet1 = writer.sheets['Loan_Cheat_Sheet']
worksheet2 = writer.sheets['Expanded']
worksheet3 = writer.sheets['Limited']
worksheet4 = writer.sheets['QCD']
worksheet5 = writer.sheets['Warehouse']

cell_format = workbook.add_format({'border': 1, 'border_color': 'black', 'font_name': 'calibri', 'font_size': 10, 'align': 'center', 'valign': 'vcenter'})

worksheet1.set_column('A:J', None, cell_format)
worksheet2.set_column('A:J', None, cell_format)
worksheet3.set_column('A:J', None, cell_format)
worksheet4.set_column('A:J', None, cell_format)
worksheet5.set_column('A:J', None, cell_format)

worksheet1.set_header(header1)
worksheet2.set_header(header2)
worksheet3.set_header(header3)
worksheet4.set_header(header4)
worksheet5.set_header(header5)

header_format = workbook.add_format({'bold': True})

writer.save()

__Step 8: Create security cheat sheet__

In [ ]:
df_seccs = df_seccs.sort_values(["COLLATERAL_PRIORITY", "COLLATERAL_ACE_CODE", "FHLB_RATING"], ascending=[True, True, False])

#insert empty row at each change of ACE_Group
df_seccs = df_seccs.reset_index(drop=True) #securities
mask = df_seccs["COLLATERAL_GROUP_CODE"].ne(df_seccs["COLLATERAL_GROUP_CODE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_seccs.columns)
df_seccs = pd.concat([df_seccs, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_seccs = df_seccs.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

#write security cheat sheet to .xlsx
cols_seccs = ["COLLATERAL_ACE_CODE", "ACE_CODE_DESC", "FHLB_RATING", "BLANKET_CL_A", "SPECIFIC_CL_A", "SPECIFIC_CL_B", "MDA_REVIEW_REQ", "CASE_BY_CASE_FEE"]

header1 = '&LSecurity_Cheat_Sheet'

writer = pd.ExcelWriter("J:\\EUA\\CRA\\Collateral Review\\Volatility Factor\\HIST\\2023\\output\\Security_Cheat_Sheet_FinSvcs.xlsx", engine='xlsxwriter')
df_seccs[cols_seccs].to_excel(writer, sheet_name='Security_Cheat_Sheet')

workbook = writer.book
worksheet1 = writer.sheets['Security_Cheat_Sheet']

cell_format = workbook.add_format({'border': 1, 'border_color': 'black', 'font_name': 'calibri', 'font_size': 10, 'align': 'center', 'valign': 'vcenter'})

worksheet1.set_column('A:K', None, cell_format)

worksheet1.set_header(header1)

header_format = workbook.add_format({'bold': True})

writer.save()

__Step 9: Create exhibit A_MPP__

In [ ]:
#duplicate df for table tabs
df_MPP_exp = df_MPP.loc[(df_MPP["COLLATERAL_TABLE"] == "Expanded") & (df_MPP["MPP_ASSET_TYPE"].notna())].copy()
df_MPP_ltd = df_MPP.loc[(df_MPP["COLLATERAL_TABLE"] == "Limited") & (df_MPP["MPP_ASSET_TYPE"].notna())].copy()
df_MPP_qcd = df_MPP.loc[(df_MPP["COLLATERAL_TABLE"] == "QCD") & (df_MPP["MPP_ASSET_TYPE"].notna())].copy()
df_MPP_sec = df_MPP.loc[(df_MPP["COLLATERAL_TABLE"] == "Securities") & (df_MPP["MPP_ASSET_TYPE"].notna())].copy()

#replace LV for MPP
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_SUB_TYPE"] == "Not Agency-eligible", "PROPOSED_LV_CL_A"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Agricultural real estate", "PROPOSED_LV_CL_A"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Second mortgages on 1-4 family residential real property", "PROPOSED_LV_CL_A"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Home Equity Line of Credit (HELOC)", "PROPOSED_LV_CL_A"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Residential construction mortgages", "PROPOSED_LV_CL_A"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Multifamily construction mortgages", "PROPOSED_LV_CL_A"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Commercial construction mortgages", "PROPOSED_LV_CL_A"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Non-lead lender participation loans", "PROPOSED_LV_CL_A"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Other real estate-related property", "PROPOSED_LV_CL_A"] = "Case-by-Case"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Operating loans (CFI only)", "PROPOSED_LV_CL_A"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Equipment loans (CFI only)", "PROPOSED_LV_CL_A"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Student loans", "PROPOSED_LV_CL_A"] = "Not applicable"

df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_SUB_TYPE"] == "Not Agency-eligible", "PROPOSED_LV_CL_B"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Agricultural real estate", "PROPOSED_LV_CL_B"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Second mortgages on 1-4 family residential real property", "PROPOSED_LV_CL_B"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Home Equity Line of Credit (HELOC)", "PROPOSED_LV_CL_B"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Residential construction mortgages", "PROPOSED_LV_CL_B"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Multifamily construction mortgages", "PROPOSED_LV_CL_B"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Commercial construction mortgages", "PROPOSED_LV_CL_B"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Non-lead lender participation loans", "PROPOSED_LV_CL_B"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Other real estate-related property", "PROPOSED_LV_CL_B"] = "Case-by-Case"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Operating loans (CFI only)", "PROPOSED_LV_CL_B"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Equipment loans (CFI only)", "PROPOSED_LV_CL_B"] = "Not applicable"
df_MPP_exp.loc[df_MPP_exp["MPP_ASSET_TYPE"] == "Student loans", "PROPOSED_LV_CL_B"] = "Not applicable"

df_MPP_ltd.loc[df_MPP_ltd["MPP_ASSET_TYPE"] == "Multifamily construction mortgages", "PROPOSED_LV_CL_A"] = 0
df_MPP_ltd.loc[df_MPP_ltd["MPP_ASSET_TYPE"] == "Commercial construction mortgages", "PROPOSED_LV_CL_A"] = 0
df_MPP_ltd.loc[df_MPP_ltd["MPP_ASSET_TYPE"] == "Non-lead lender participation loans", "PROPOSED_LV_CL_A"] = "Applicable asset LV less 10%"
df_MPP_ltd.loc[df_MPP_ltd["MPP_ASSET_TYPE"] == "Other real estate-related property", "PROPOSED_LV_CL_A"] = "Case-by-Case"

df_MPP_qcd.loc[df_MPP_qcd["MPP_ASSET_TYPE"] == "Multifamily construction mortgages", "PROPOSED_LV_CL_A"] = 0
df_MPP_qcd.loc[df_MPP_qcd["MPP_ASSET_TYPE"] == "Commercial construction mortgages", "PROPOSED_LV_CL_A"] = 0
df_MPP_qcd.loc[df_MPP_qcd["MPP_ASSET_TYPE"] == "Non-lead lender participation loans", "PROPOSED_LV_CL_A"] = "Applicable asset LV less 10%"
df_MPP_qcd.loc[df_MPP_qcd["MPP_ASSET_TYPE"] == "Other real estate-related property", "PROPOSED_LV_CL_A"] = "Case-by-Case"

df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_TYPE"] == "Mutual Funds", "PROPOSED_LV_CL_A"] = "Case-by-Case"
df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_TYPE"] == "State and local government securities", "PROPOSED_LV_CL_A"] = "Case-by-Case"
df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_SUB_TYPE"] == "PLMBS - HE", "PROPOSED_LV_CL_A"] = "Case-by-Case"
df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_SUB_TYPE"] == "PLMBS - Inverse floater", "PROPOSED_LV_CL_A"] = "Case-by-Case"
df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_SUB_TYPE"] == "PLMBS - SUP, MEZZ", "PROPOSED_LV_CL_A"] = "Case-by-Case"

df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_TYPE"] == "Mutual Funds", "PROPOSED_LV_CL_B"] = "Case-by-Case"
df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_TYPE"] == "State and local government securities", "PROPOSED_LV_CL_B"] = "Case-by-Case"
df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_SUB_TYPE"] == "PLMBS - HE", "PROPOSED_LV_CL_B"] = "Case-by-Case"
df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_SUB_TYPE"] == "PLMBS - Inverse floater", "PROPOSED_LV_CL_B"] = "Case-by-Case"
df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_SUB_TYPE"] == "PLMBS - SUP, MEZZ", "PROPOSED_LV_CL_B"] = "Case-by-Case"

#replace rating for MPP
df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_TYPE"] == "Commercial Mortgage-Backed Securities - 100% defeased (rated at least equivalent to the U.S. Government)", "FHLB_RATING"] = "NR"
df_MPP_sec.loc[df_MPP_sec["MPP_ASSET_TYPE"] == "State and local government securities", "FHLB_RATING"] = "NR"

#MPP LV column names
df_MPP_exp["DELIVERED_EXPANDED_LV_CL_A"] = df_MPP_exp["PROPOSED_LV_CL_A"]
df_MPP_exp["DELIVERED_EXPANDED_LV_CL_B"] = df_MPP_exp["PROPOSED_LV_CL_B"]

df_MPP_ltd["DELIVERED_LIMITED_LV"] = df_MPP_ltd["PROPOSED_LV_CL_A"]

df_MPP_qcd["BLANKET_QCD_LV"] = df_MPP_qcd["PROPOSED_LV_CL_A"]

df_MPP_sec["BLANKET_QCD_LV"] = "Not applicable"
df_MPP_sec["DELIVERED_LIMITED_LV"] = df_MPP_sec["PROPOSED_LV_CL_A"]
df_MPP_sec["DELIVERED_EXPANDED_LV_CL_A"] = df_MPP_sec["PROPOSED_LV_CL_A"]
df_MPP_sec["DELIVERED_EXPANDED_LV_CL_B"] = df_MPP_sec["PROPOSED_LV_CL_B"]

#merge loan dataframes
df_MPP_loan = df_MPP_ltd.merge(df_MPP_qcd[["MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE", "BLANKET_QCD_LV"]], on=["MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE"], how="left") #ltd and qcd
df_MPP_loan = df_MPP_loan.merge(df_MPP_exp[["MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE", "DELIVERED_EXPANDED_LV_CL_A", "DELIVERED_EXPANDED_LV_CL_B"]], on=["MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE"], how="left") #ltd, qcd and exp

#keep only columns needed for MPP
df_MPP_loan = df_MPP_loan[["MPP_SORT", "MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE", "FHLB_RATING", "BLANKET_QCD_LV", "DELIVERED_LIMITED_LV", "DELIVERED_EXPANDED_LV_CL_A", "DELIVERED_EXPANDED_LV_CL_B"]]

df_MPP_sec = df_MPP_sec[["MPP_SORT", "MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE", "FHLB_RATING", "BLANKET_QCD_LV", "DELIVERED_LIMITED_LV", "DELIVERED_EXPANDED_LV_CL_A", "DELIVERED_EXPANDED_LV_CL_B"]]

#stack loan and security dataframes
df_MPP_a = pd.concat([df_MPP_loan, df_MPP_sec], ignore_index=True, axis=0)

#sort dataframes in MPP order
df_MPP_a = df_MPP_a.sort_values(["MPP_SORT", "MPP_ASSET_SUB_TYPE", "FHLB_RATING"], ascending=[True, True, False])

#insert empty row at each change of ASSET_TYPE
df_MPP_a = df_MPP_a.reset_index(drop=True) #exhibit a
mask = df_MPP_a["MPP_ASSET_TYPE"].ne(df_MPP_a["MPP_ASSET_TYPE"].shift(-1))
df3 = pd.DataFrame(data=None, index=mask.index[mask] + .5, columns=df_MPP_a.columns)
df_MPP_a = pd.concat([df_MPP_a, df3]).sort_index().reset_index(drop=True).iloc[:-1]

df_MPP_a = df_MPP_a.set_index(["MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE", "FHLB_RATING"])

#write exhibit a to .xlsx
cols_a = ["BLANKET_QCD_LV", "DELIVERED_LIMITED_LV", "DELIVERED_EXPANDED_LV_CL_A", "DELIVERED_EXPANDED_LV_CL_B"]

header1 = '&LExhibit_A_MPP'

writer = pd.ExcelWriter("J:\\EUA\\CRA\\Collateral Review\\Volatility Factor\\HIST\\2023\\output\\Exhibit_A_MPP.xlsx", engine='xlsxwriter')
df_MPP_a[cols_a].to_excel(writer, sheet_name='Exhibit_A')

workbook = writer.book
worksheet1 = writer.sheets['Exhibit_A']

cell_format = workbook.add_format({'border': 1, 'border_color': 'black', 'font_name': 'calibri', 'font_size': 10, 'align': 'center', 'valign': 'vcenter'})

worksheet1.set_column('A:G', None, cell_format)

worksheet1.set_header(header1)

header_format = workbook.add_format({'bold': True})

writer.save()